In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

*예상 사용 시간: Eagle r3 프로세서 기준 1분 이내 (참고: 이는 예상치이며, 실제 실행 시간은 다를 수 있습니다.)*

## 배경

더 빠르고 효율적인 결과를 보장하기 위해, 2024년 3월 1일부터 Circuit과 옵저버블은 Qiskit Runtime 프리미티브에 제출하기 전에 QPU(양자 처리 장치)가 지원하는 명령어만 사용하도록 변환되어야 합니다. 이를 *명령어 집합 아키텍처*(ISA) Circuit 및 옵저버블이라고 합니다. 이를 수행하는 일반적인 방법 중 하나는 Transpiler의 `generate_preset_pass_manager` 함수를 사용하는 것입니다. 그러나 더 수동적인 방법을 따를 수도 있습니다.

예를 들어, 특정 장치의 특정 Qubit 서브셋을 대상으로 지정하고 싶을 수 있습니다. 이 연습에서는 Circuit을 생성하고, transpile하고, 제출하는 전체 과정을 완료하여 다양한 Transpiler 설정의 성능을 테스트합니다.

## 요구사항

시작하기 전에 다음이 설치되어 있는지 확인하세요:

* Qiskit SDK v1.2 이상, [시각화](https://docs.quantum.ibm.com/api/qiskit/visualization) 지원 포함
* Qiskit Runtime v0.28 이상 (`pip install qiskit-ibm-runtime`)

## 설정

In [1]:
# Create circuit to test transpiler on
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import grover_operator, DiagonalGate

# Use Statevector object to calculate the ideal output
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram
from qiskit.transpiler import PassManager

from qiskit.circuit.library import XGate
from qiskit.quantum_info import hellinger_fidelity

## 1단계: 고전적 입력을 양자 문제로 매핑
Transpiler가 최적화를 시도할 작은 Circuit을 생성합니다. 이 예제에서는 상태 `111`을 표시하는 오라클을 사용하여 Grover 알고리즘을 수행하는 Circuit을 생성합니다. 다음으로, 나중에 비교하기 위해 이상적인 분포(완벽한 양자 컴퓨터에서 무한히 실행했을 때 측정할 것으로 예상되는 값)를 시뮬레이션합니다.

In [2]:
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

qc.draw(output="mpl", style="iqp")

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ac958d4-b9b5-4939-a359-a9edca7ddb6a-0.svg" alt="Output of the previous code cell" />

In [3]:
ideal_distribution = Statevector.from_instruction(qc).probabilities_dict()

plot_histogram(ideal_distribution)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/6313186e-bc40-432e-9ada-8594d6a26d55-0.svg" alt="Output of the previous code cell" />

## Transpile

Next, transpile the circuits for the QPU. You will compare the performance of the transpiler with `optimization_level` set to `0` (lowest) against `3` (highest). The lowest optimization level does the bare minimum needed to get the circuit running on the device; it maps the circuit qubits to the device qubits and adds swap gates to allow all two-qubit operations. The highest optimization level is much smarter and uses lots of tricks to reduce the overall gate count. Since multi-qubit gates have high error rates and qubits decohere over time, the shorter circuits should give better results.

<Admonition type="important">
    This example uses IBM Quantum&reg; hardware, but you can try it on any Qiskit-compatible QPU.  Your results might be different.
</Admonition>

The following cell transpiles `qc` for both values of `optimization_level`, prints the number of two-qubit gates, and adds the transpiled circuits to a list. Some of the transpiler's algorithms are randomized, so it sets a seed for reproducibility.

In [4]:
# Use Qiskit Runtime to run jobs on hardware
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    SamplerV2 as Sampler,
)

In [5]:
# Select the backend with the fewest number of jobs in the queue
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
backend.name

'ibm_marrakesh'

In [6]:
# Need to add measurements to the circuit
qc.measure_all()

# Find the correct two-qubit gate
twoQ_gates = set(["ecr", "cz", "cx"])
for gate in backend.basis_gates:
    if gate in twoQ_gates:
        twoQ_gate = gate

circuits = []
for optimization_level in [0, 3]:
    pm = generate_preset_pass_manager(
        optimization_level, backend=backend, seed_transpiler=0
    )
    t_qc = pm.run(qc)
    print(
        f"Two-qubit gates (optimization_level={optimization_level}): ",
        t_qc.count_ops()[twoQ_gate],
    )
    circuits.append(t_qc)

Two-qubit gates (optimization_level=0):  21
Two-qubit gates (optimization_level=3):  12


Since CNOTs usually have a high error rate, the circuit transpiled with `optimization_level=3` should perform much better.

Another way you can improve performance is through [dynamical decoupling](/docs/api/qiskit/qiskit.transpiler.passes.PadDynamicalDecoupling), by applying a sequence of gates to idling qubits. This cancels out some unwanted interactions with the environment. The following cell adds dynamic decoupling to the circuit transpiled with `optimization_level=3` and adds it to the list.

In [7]:
from qiskit_ibm_runtime.transpiler.passes.scheduling import (
    ASAPScheduleAnalysis,
    PadDynamicalDecoupling,
)

# Get gate durations so the transpiler knows how long each operation takes
durations = backend.target.durations()

# This is the sequence we'll apply to idling qubits
dd_sequence = [XGate(), XGate()]

# Run scheduling and dynamic decoupling passes on circuit
pm = PassManager(
    [
        ASAPScheduleAnalysis(durations),
        PadDynamicalDecoupling(durations, dd_sequence),
    ]
)
circ_dd = pm.run(circuits[1])

# Add this new circuit to our list
circuits.append(circ_dd)

In [8]:
circ_dd.draw(output="mpl", style="iqp", idle_wires=False)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/c1c91fbd-acfe-413e-a6c9-ad97f4dd5543-0.svg" alt="Output of the previous code cell" />

## Execute the circuit

At this point, you have a list of circuits transpiled with different settings. Next, run these circuits using the Sampler primitive and store the results to `result`.

In [9]:
sampler = Sampler(backend)
job = sampler.run(
    [(circuit) for circuit in circuits],  # sample all three circuits
    shots=8000,
)
result = job.result()

![이전 코드 셀의 출력](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/4ada6498-b9d7-4d88-b8a9-ef1dc0a85bf7-0.avif)

## 3단계: Qiskit 프리미티브를 사용한 실행
이 시점에서 지정된 QPU를 위해 transpile된 Circuit 목록이 있습니다. 다음으로, Sampler 프리미티브 인스턴스를 생성하고 배치 작업을 시작합니다. 컨텍스트 매니저(`with ...:`)를 사용하면 배치가 자동으로 열리고 닫힙니다.

컨텍스트 매니저 내에서 Circuit을 샘플링하고 결과를 `result`에 저장합니다.

In [10]:
binary_prob = [
    {
        k: v / res.data.meas.num_shots
        for k, v in res.data.meas.get_counts().items()
    }
    for res in result
]
plot_histogram(
    binary_prob + [ideal_distribution],
    bar_labels=False,
    legend=[
        "optimization_level=0",
        "optimization_level=3",
        "optimization_level=3 + dd",
        "ideal distribution",
    ],
)

<Image src="../docs/images/guides/circuit-transpilation-settings/extracted-outputs/9e86132d-a8b2-40db-af42-53042dfa108b-0.svg" alt="Output of the previous code cell" />

## 4단계: 후처리 및 원하는 고전적 형식으로 결과 반환
마지막으로, 장치 실행 결과를 이상적인 분포와 비교하여 플롯합니다. Gate 수가 적기 때문에 `optimization_level=3`의 결과가 이상적인 분포에 더 가깝고, 동적 디커플링으로 인해 `optimization_level=3 + dd`는 더욱 가까운 것을 확인할 수 있습니다.

In [11]:
for prob in binary_prob:
    print(f"{hellinger_fidelity(prob, ideal_distribution):.3f}")

0.985
0.989
0.988


![이전 코드 셀의 출력](../docs/images/guides/circuit-transpilation-settings/extracted-outputs/525777ea-d438-4f3b-acb6-53e579f24a0e-0.avif)

각 결과 세트와 이상적인 분포 사이의 [Hellinger 충실도](https://docs.quantum.ibm.com/api/qiskit/quantum_info)를 계산하여 이를 확인할 수 있습니다(높을수록 좋으며, 1이 완벽한 충실도입니다).